# 🏠 房价预测：基于 PyTorch 的单变量线性回归

湖北理工学院《人工智能理论》课程资料

📝 **本节内容概述：**
1. 绕不开的经典问题：准备波士顿房价数据
2. 💡 核心理论：模型、损失函数与梯度下降
3. 🪄 PyTorch 魔法：利用自动求导实现前向反向传播

## 📊 1. 获取房价数据

首先，我们将读取本地整理好的波士顿房价数据 (`./data/houseprice.csv`)。

原始数据包含 13 种描述房屋和周边环境的特征 (Features) 和 1 个我们想要预测的目标房价 (Target)。这也是深度学习中最经典的回归问题数据集。

**特征字典 (Features)：**
- `[0]`: 按城镇划分的犯罪率
- `[1]`: 面积超过25,000平方英尺（2322.58平方米）的住宅用地占比
- `[2]`: 非零售商业用地占比
- `[3]`: 是否位于查尔斯河畔 (1为是，0为否)
- `[4]`: 氮氧化物浓度（空气质量） (单位：ppm)
- `[5]`: **每个住宅的平均房间数** 👉 **(我们将使用这个作为输入 x)**
- `[6]`: 1940年以前建造的自住房比例
- `[7]`: 到波士顿五个就业中心的加权距离
- `[8]`: 辐射路可达性指数
- `[9]`: 每10,000美元的房产税率
- `[10]`: 学生-教师比率
- `[11]`: 黑人城镇比例相关的统计指标
- `[12]`: 低层次人口占比

**目标预测标量 (Target)：**
- **自住房房价中位数（单位：千美元）** 👉 **(我们将使用这个作为预测目标 y)**

在本实验中，为了方便理解线性回归的核心思想并直观地画出二维图像，我们将提取**单一特征 `[5]` (平均房间数)** 去预测目标值 **房价中位数**。这也是为什么后续代码中会看到 `x_np = data[:, 5]` 和 `y_np = target`。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

# ========================================================
# 👇 设置中文字体，确保图表正常显示汉字
# ========================================================
plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti SC', 'STHeiti', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# ========================================================
# 👇 导入数据
# ========================================================
data_url = './data/houseprice.csv'
raw_df = pd.read_csv(data_url, sep=',', skiprows=1, header=None)

# 提取特征矩阵和目标标签
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, 1:2]])
target = raw_df.values[1::2, 2]

# 提取单一特征：[5] 对应的是平均房间数
x_np = data[:, 5]
y_np = target

print(f"✅ 数据加载完成！样本总数: {len(x_np)}")
print(f"示例：房间数 = {x_np[0]:.2f}, 房价 = {y_np[0]:.2f}千美元")

# 绘制原始数据散点图
plt.figure(figsize=(8, 5))
plt.scatter(x_np, y_np, color='dodgerblue', alpha=0.6, edgecolors='k')
plt.xlabel('平均房间数', fontsize=12)
plt.ylabel('房价中位数（千美元）', fontsize=12)
plt.title('房间数与房价的关系', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 🧠 2. 核心理论速览

在构建预测模型之前，必须理解深度学习大厦的三块核心基石。

> 💡 **知识点**

### 1️⃣ 什么是模型？(Model)
在线性回归中，我们假设数据的关系是一条直线，可以用公式表达：
$$\hat{y} = w \cdot x + b$$
其中，$\hat{y}$ 是预测房价，$x$ 是房间数。我们的最终目的就是找到完美的斜率 $w$（Weight）和截距 $b$（Bias）。

### 2️⃣ 什么是损失函数？(Loss Function)
我们有了一组 $w$ 和 $b$，但怎么知道它好不好呢？**均方误差 (MSE)** 是评价线性回归好坏的常见标准之一。
$$ Loss = \frac{1}{2N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2 $$
损失值越小，代表预测线距离所有的离散数据点越近，模型越精准。我们称计算 Loss 的过程为 **前向传播 (Forward Pass)**。

### 3️⃣ 什么是梯度下降？(Gradient Descent)
面对山坡上的一个球（Loss），想让球滚到谷底（Loss等于0），最好的办法就是**一直顺着最陡峭的方向走**。梯度，在数学上衡量了陡峭程度，也就是函数的导数。

通过计算 Loss 对每一项参数的导数 $\frac{\partial L}{\partial w}$, $\frac{\partial L}{\partial b}$（这个过程被称为 **反向传播 Backward Pass**），我们使用如下公式不断更新参数，就能让误差越来越小：
$$ w = w - \eta \cdot \frac{\partial L}{\partial w} $$
这里的 $\eta$ 是学习率，决定了每一步迈得多大。

## 3. 拆分数据集

在真实的机器学习项目中，我们不能拿训练数据来评判模型的好坏。因此，需要将全部数据拆分为三份：

- **训练集 (Train)**: 喂给模型去学习
- **验证集 (Validation)**: 每轮训练结束后，衡量模型在未见过的数据上表现如何
- **测试集 (Test)**: 模型训练完毕后做最终评估

In [ ]:
from sklearn.model_selection import train_test_split

# ========================================================
# 👇 拆分数据集：60% 训练 / 20% 验证 / 20% 测试
# ========================================================
x_train_np, x_temp_np, y_train_np, y_temp_np = train_test_split(
    x_np, y_np, test_size=0.4, random_state=42)
x_val_np, x_test_np, y_val_np, y_test_np = train_test_split(
    x_temp_np, y_temp_np, test_size=0.5, random_state=42)

print(f'训练集大小: {len(x_train_np)}')
print(f'验证集大小: {len(x_val_np)}')
print(f'测试集大小: {len(x_test_np)}')

## 4. PyTorch 训练 + 实时可视化

PyTorch 的 **Autograd** 让我们不需要手工推导导数公式，只需设置 `requires_grad=True`，反向传播就全自动完成了。

下面的代码将一边训练，一边动态刷新 **2 x 3** 的图表矩阵：
- **第一行**: 训练集、验证集、测试集上的拟合效果
- **第二行**: 训练集、验证集、测试集上的损失变化曲线

In [ ]:
import torch
from IPython.display import clear_output, display

# ========================================================
# 1. 准备数据
# ========================================================
x_train_t = torch.tensor(x_train_np, dtype=torch.float32)
y_train_t = torch.tensor(y_train_np, dtype=torch.float32)
x_val_t   = torch.tensor(x_val_np,   dtype=torch.float32)
y_val_t   = torch.tensor(y_val_np,   dtype=torch.float32)
x_test_t  = torch.tensor(x_test_np,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test_np,  dtype=torch.float32)

# ========================================================
# 2. 初始化参数
# ========================================================

# 给予固定的起始位置，保证复现一致性

w = torch.tensor([-50.0], requires_grad=True)
b = torch.tensor([50.0], requires_grad=True)

learning_rate = 0.03
epochs = 5000
plot_every = 100  # 每隔多少轮刷新一次图表

# 记录损失历史
history_epochs     = []
history_train_loss = []
history_val_loss   = []
history_test_loss  = []

def compute_loss(x, y, w, b):
    with torch.no_grad():
        return ((w * x + b - y) ** 2).mean().item() / 2

# 图表配色和标题
colors      = ['dodgerblue', 'darkorange', 'forestgreen']
fit_titles  = ['Train: fitting', 'Val: fitting', 'Test: fitting']
loss_titles = ['Train: loss', 'Val: loss', 'Test: loss']

# 统一的 x 范围，用于绘制回归线
x_all = np.concatenate([x_train_np, x_val_np, x_test_np])
x_line = np.array([x_all.min() - 0.5, x_all.max() + 0.5])

datasets = [
    (x_train_np, y_train_np),
    (x_val_np,   y_val_np),
    (x_test_np,  y_test_np),
]

# ========================================================
# 3. 训练循环 + 实时 2x3 可视化
# ========================================================
for epoch in range(epochs):
    # --- 前向传播 (仅训练集) ---
    y_pred = w * x_train_t + b
    loss = ((y_pred - y_train_t) ** 2).mean() / 2
    
    # --- 反向传播 ---
    loss.backward()
    
    # --- 梯度下降 + 清零 ---
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        w.grad.zero_()
        b.grad.zero_()
    
    # --- 👇 每 plot_every 轮或最后一轮刷新图表，复杂的画图可以不用读 ---
    if (epoch + 1) % plot_every == 0 or epoch == 0 or epoch == epochs - 1:
        train_loss = compute_loss(x_train_t, y_train_t, w, b)
        val_loss   = compute_loss(x_val_t,   y_val_t,   w, b)
        test_loss  = compute_loss(x_test_t,  y_test_t,  w, b)
        
        history_epochs.append(epoch + 1)
        history_train_loss.append(train_loss)
        history_val_loss.append(val_loss)
        history_test_loss.append(test_loss)
        
        losses_list = [history_train_loss, history_val_loss, history_test_loss]
        
        # 清除旧输出，实现动画效果
        clear_output(wait=True)
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(
            f'Epoch {epoch+1}/{epochs}   '
            f'w={w.item():.4f}  b={b.item():.4f}   '
            f'Train Loss={train_loss:.4f}  '
            f'Val Loss={val_loss:.4f}  '
            f'Test Loss={test_loss:.4f}',
            fontsize=14
        )
        
        y_line = w.item() * x_line + b.item()
        
        # --- 第一行：拟合效果 ---
        for col in range(3):
            ax = axes[0][col]
            xd, yd = datasets[col]
            ax.scatter(xd, yd, color=colors[col], alpha=0.6, edgecolors='k', s=30, label='data')
            ax.plot(x_line, y_line, color='crimson', linewidth=2.5,
                    label=f'y={w.item():.2f}x + {b.item():.2f}')
            ax.set_title(fit_titles[col], fontsize=13)
            ax.set_xlabel('Rooms')
            ax.set_ylabel('Price')
            ax.legend(fontsize=9)
            ax.grid(True, linestyle='--', alpha=0.4)
        
        # --- 第二行：损失曲线 ---
        for col in range(3):
            ax = axes[1][col]
            lh = losses_list[col]
            ax.plot(history_epochs, lh, color=colors[col], linewidth=2, marker='o', markersize=3)
            ax.set_title(loss_titles[col], fontsize=13)
            ax.set_xlabel('Epoch')
            ax.set_ylabel('MSE Loss')
            if len(lh) > 1 and max(lh) / (min(lh) + 1e-8) > 10:
                ax.set_yscale('log')
            ax.grid(True, linestyle='--', alpha=0.4)
        
        plt.tight_layout(rect=[0, 0, 1, 0.94])
        plt.show()

print(f'Training done!  w = {w.item():.4f}, b = {b.item():.4f}')